# 브랜드 자동 리포트 데이터 정합성 감사

## tl;dr

- 원천 재구성 합계, 국가·SKU 요약, 월별 큐브, 월 커버리지, 브랜드 리포트의 **매출 €36,197,522.84와 유상 판매수량 5,743,900개가 전부 일치**한다.
- 편강율 화면값도 원천 집계와 일치한다: **매출 €41,361.75, 판매수량 8,072개, 판매 SKU 13개, 최고월 2024-09 €14,358.84**.
- 현재 재고 스냅샷은 2,586개 SKU가 모두 유일하고, 기준일은 2026-07-23 하나이며, 일평균 판매량은 `최근 3개월 판매수량 ÷ 90`과 전 행 일치한다.
- 원화 환산은 **현재 적용 환율 EUR/KRW 1,689.48(고시일 2026-07-23)**을 사용하고, 화면에도 현재 환율임을 명시한다.
- 판매금액 0·수량 양수인 무상증정 467행, 33,742개는 판매 집계에서 제외했다.
- 원천에는 완전 중복 후보 63쌍(분석 대상 기준 최대 매출 0.1694% 영향)과 부분 월 2024-04이 존재한다.


## Context & Methods

### 감사 범위

- 판매 분석: PL 법인, 2024-04-01~2024-12-31, CMS `/eu/sales/local`
- 판매 원천 단위: 송장·SKU 출고행
- 브랜드 리포트 단위: 브랜드 합계와 브랜드별 국가·SKU·카테고리·월 집계
- 재고 단위: 2026-07-23 기준 SKU별 발주분석 스냅샷

### Key Assumptions

- `amount_krw` 필드는 현재 API에서 EUR 정규화 금액으로 사용된다는 애플리케이션 정의를 따른다.
- 원천에 행 식별자가 없어 완전 중복 후보가 실제 중복인지 동일 송장의 합법적인 반복 라인인지는 별도 소스 확인이 필요하다.
- 매출 0 출고를 판매수량에 포함할지는 업무 정의가 필요하므로 오류로 단정하지 않고 정책 이슈로 분류한다.


## Data

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
sys.path.insert(0, str(project_root / "artifacts"))
from brand_report_data_quality_audit import run_audit

audit = run_audit()
print("판매 원천:", audit["source_path"])
print("시즌 스냅샷:", audit["season_snapshot"])
print("재고 스냅샷:", audit["order_snapshot"])
print("분석 옵션:", {
    key: audit["analysis_options"].get(key)
    for key in ("start_date", "end_date", "entity_code", "average_eur_krw_rate", "exchange_rate_date")
})

판매 원천: backend\storage\cms_fetch_cache\46f5ffcebf51f45b836b63c127f9f8ce929168e74d06a718204445a525651cc6.json
시즌 스냅샷: backend\storage\latest_season_trend\latest_adminmaster__PL.json
재고 스냅샷: backend\storage\latest_order_review\adminmaster__PL__06ef365e-3b27-4980-b416-f4627440f520.json
분석 옵션: {'start_date': '2024-04-01', 'end_date': '2024-12-31', 'entity_code': 'PL', 'average_eur_krw_rate': 1689.48, 'exchange_rate_date': '2026-07-23'}


In [2]:
source_summary = pd.DataFrame([
    {"항목": "판매 원천 행", "값": audit["raw_profile"]["rows"]},
    {"항목": "최종 리포트 모집단 행", "값": audit["row_flow"]["final_rows"]},
    {"항목": "브랜드 수", "값": audit["report_checks"]["brand_count"]},
    {"항목": "재고 SKU 수", "값": audit["stock_profile"]["rows"]},
    {"항목": "완료 월", "값": audit["temporal_profile"]["complete_months"]},
    {"항목": "부분 월", "값": ", ".join(audit["temporal_profile"]["partial_months"])},
])
display(source_summary)

,항목,값
0,판매 원천 행,24995
1,최종 리포트 모집단 행,22106
2,브랜드 수,22
3,재고 SKU 수,2586
4,완료 월,8
5,부분 월,2024-04


## Results

### 1. 필수 필드 완전성

In [3]:
display(audit["completeness"].sort_values("missing_rate_pct", ascending=False))
print("상품 마스터 결합:", audit["mapping_profile"])

,column,missing_rows,missing_rate_pct
3,brand_nm,794,3.1766
1,prod_cd,0,0.0000
0,invc_no,0,0.0000
2,prod_nm,0,0.0000
4,qty,0,0.0000
5,amount,0,0.0000
6,amount_krw,0,0.0000
7,curr,0,0.0000
8,ship_dt,0,0.0000
9,biz_type,0,0.0000


상품 마스터 결합: {'product_rows': 2924, 'distinct_product_codes': 2924, 'duplicate_product_code_rows': 0, 'final_sales_rows': 22106, 'unmatched_master_rows': 0, 'unmatched_master_qty_share_pct': 0.0, 'unmatched_master_amount_share_pct': 0.0, 'unmapped_category_rows': 8, 'missing_brand_rows': 0, 'missing_country_rows': 0}


### 2. 원천→집계→리포트 합계 대사

In [4]:
display(audit["totals"].round(6))
checks = pd.DataFrame([
    {"검사": key, "통과": value}
    for key, value in audit["report_checks"].items()
])
display(checks)

,layer,amount,qty,amount_diff_vs_source,qty_diff_vs_source
0,reconstructed_source,36197522.84,5743900.0,0.0,0.0
1,countrySkuSummary,36197522.84,5743900.0,-0.0,0.0
2,countrySkuMonthly,36197522.84,5743900.0,0.0,0.0
3,monthCoverage,36197522.84,5743900.0,-0.0,0.0
4,brandReports,36197522.84,5743900.0,-0.0,0.0


,검사,통과
0,cached_reports_match_source_guard,True
1,cached_reports_equal_rebuild,True
2,source_summary_amount_match,True
3,source_summary_qty_match,True
4,summary_monthly_amount_match,True
5,summary_monthly_qty_match,True
6,monthly_coverage_amount_match,True
7,monthly_coverage_qty_match,True
8,summary_report_amount_match,True
9,summary_report_qty_match,True


### 3. 편강율 화면값 대사

In [5]:
focus_sales = pd.DataFrame(audit["focus"]["sales"])
focus_stock = pd.DataFrame(audit["focus"]["stock"])
display(focus_sales[[
    "브랜드", "source_amount", "report_amount", "amount_diff",
    "source_qty", "report_qty", "qty_diff", "source_skus", "report_skus",
    "monthly_diff", "top5_share_pct"
]])
display(focus_stock)

,브랜드,source_amount,report_amount,amount_diff,source_qty,report_qty,qty_diff,source_skus,report_skus,monthly_diff,top5_share_pct
0,편강율,41361.75,41361.75,0.0,8062,8062.0,0.0,13,13,0.0,59.235235


,브랜드,stock_skus,available_qty,daily_sales_qty,risk_skus,moi
0,편강율,15,3474.0,144.466667,12,0.801569


### 4. 월 커버리지와 부분 월

In [6]:
display(audit["month_profile"])
print("부분 월 매출 비중(%):", audit["temporal_profile"]["partial_month_amount_share_pct"])

,month,status,rowCount,activeDays,expectedBusinessDays,activityRatio,firstDate,lastDate,totalAmount,totalQty
0,2024-04,partial,59,4,22,0.1818,2024-04-11,2024-04-30,434504.78,75390.0
1,2024-05,complete,1447,20,23,0.8696,2024-05-02,2024-05-31,2812415.95,427929.0
2,2024-06,complete,1730,20,20,1.0000,2024-06-03,2024-06-28,4385320.60,707545.0
3,2024-07,complete,2207,23,23,1.0000,2024-07-01,2024-07-31,4617590.68,730787.0
4,2024-08,complete,2286,20,22,0.9091,2024-08-01,2024-08-30,4475740.50,733138.0
5,2024-09,complete,3310,21,21,1.0000,2024-09-02,2024-09-30,5292880.94,828026.0
6,2024-10,complete,3701,23,23,1.0000,2024-10-01,2024-10-31,5951199.60,961361.0
7,2024-11,complete,3668,20,21,0.9524,2024-11-04,2024-11-29,4529117.50,713458.0
8,2024-12,complete,3698,18,22,0.8182,2024-12-02,2024-12-30,3698752.29,566266.0


부분 월 매출 비중(%): 1.2004


### 5. 재고·MOI 결합 품질

In [7]:
stock_keys = [
    "rows", "distinct_skus", "duplicate_sku_rows", "missing_sku_rows",
    "missing_brand_rows", "base_dates", "negative_available_rows",
    "negative_recent_sales_rows", "daily_sales_formula_mismatch_rows",
    "sales_brand_stock_match_rate_pct", "order_required_rows",
]
display(pd.DataFrame([
    {"검사": key, "결과": audit["stock_profile"][key]}
    for key in stock_keys
]))

,검사,결과
0,rows,2586
1,distinct_skus,2586
2,duplicate_sku_rows,0
3,missing_sku_rows,0
4,missing_brand_rows,0
5,base_dates,[2026-07-23]
6,negative_available_rows,0
7,negative_recent_sales_rows,0
8,daily_sales_formula_mismatch_rows,0
9,sales_brand_stock_match_rate_pct,100.0


### 6. 원천 이상치와 표시 기준

In [8]:
anomaly_keys = [
    "duplicate_groups", "duplicate_excess_rows_if_keep_first",
    "duplicate_excess_amount_share_pct", "final_zero_amount_rows",
    "final_zero_amount_qty", "final_zero_amount_qty_share_pct",
    "final_negative_rows", "final_negative_amount_share_pct",
]
display(pd.DataFrame([
    {"검사": key, "결과": audit["anomaly_profile"][key]}
    for key in anomaly_keys
]))
display(audit["currency_ratio"])

,검사,결과
0,duplicate_groups,63.0000
1,duplicate_excess_rows_if_keep_first,63.0000
2,duplicate_excess_amount_share_pct,0.1686
3,final_zero_amount_rows,0.0000
4,final_zero_amount_qty,0.0000
5,final_zero_amount_qty_share_pct,0.0000
6,final_negative_rows,8.0000
7,final_negative_amount_share_pct,-0.0194


,currency,rows,median_ratio
0,EUR,23644,1.0
1,GBP,200,1.173847


In [9]:
findings = pd.DataFrame([
    {
        "심각도": item["severity"],
        "발견사항": item["finding"],
        "근거": str(item["evidence"]),
    }
    for item in audit["findings"]
])
display(findings)

,심각도,발견사항,근거
0,Medium,CMS 판매 원천의 완전 중복 후보,"{'affected_rows': 126, 'duplicate_groups': 63,..."
1,Low,음수 판매수량 또는 매출 행,"{'affected_rows': 8, 'net_amount': -7012.29, '..."
2,Medium,부분 월이 총매출에 포함됨,"{'months': 9, 'complete_months': 8, 'partial_m..."


## Takeaways

1. **집계 로직 정합성은 통과**다. 화면 수치가 집계 과정에서 틀어지는 문제는 현재 스냅샷에서 발견되지 않았다.
2. **원화 환산은 현재 적용 환율로 명확히 표시한다.** 환율 API를 일 1회 갱신하고, 성공 시 고시일을 함께 표시하며 실패 시에는 기본값임을 명시한다.
3. **완전 중복 후보는 CMS 원천에 존재한다.** 63쌍의 최대 영향은 분석 대상 매출의 0.1694%이며, CMS 행 ID가 없어 자동 삭제하지 않는다. 영향이 0.5% 이상이면 자동 발주 추천을 차단한다.
4. **무상증정은 판매수량에서 제외했다.** 판매금액 0·수량 양수인 467행, 33,742개를 판매 집계 모집단에서 제거했다.
5. **2024-04은 부분 월**이다. 전체 매출에는 포함되지만 YoY·추세 비교에서는 완료월만 사용해야 한다. 현재 YoY 로직은 완료월만 사용한다.
